In [ ]:
import os
import pandas as pd
from datetime import datetime
import pydicom
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import csv
import re

from utils.selection import *


# Source of RAW DICOMs
BERN_PATH='/nasdata/Berner_Daten/Other_Side_Bilddaten' 

DATA_PATH='/home/bule/berner_daten_analyse/data'
file_path =os.path.join(DATA_PATH, 'clinical','Kopie Olea von Relevent Data Other File 20181005.xlsx') ## <-- this file needs to be copied from Other Side Bildaten

data = pd.read_excel(file_path)
#remove the last empty rows -> these are just commentings
data = data.iloc[:578] 
data.tail()

check stroke date and image date

In [ ]:
data.info() # -> open in data wrangler  

In [ ]:
data

# Klinische Daten Lisa

In [ ]:

file_path = '/home/bule/berner_daten_analyse/data/clinical/lisa_stroke/data_wide_all_nihss.csv'

data_lisa = pd.read_csv(file_path)

for col in data_lisa.columns:
    print(col)

In [ ]:
data_lisa['mrs_3months_binary']=data_lisa['mrs_3months_binary'].astype('category')
data_lisa['mrs_3months_binary'].value_counts()

In [ ]:
103/(222) ~ 46% of the patients have a good outcome

# Imputation

In [ ]:
import pandas as pd
import numpy as np
from tabpfn import TabPFNClassifier

data_encoded = pd.get_dummies(data, dummy_na=True)

# Impute missing values using TabPFN
for column in data.columns:
    if data[column].isnull().any():
        print(f"Imputing column: {column}")
        
        target = data[column]
        features = data.drop(columns=[column])
        mask = target.notnull()
        
        clf = TabPFNClassifier(device='cpu')  # Use 'cuda' if GPU is available
        clf.fit(features[mask], target[mask])
        
        missing_mask = ~mask
        predictions = clf.predict(features[missing_mask])
        
        data.loc[missing_mask, column] = predictions

print(data)